# Plantilla de Proof of Concept — Curso AI Project
### De las guías al código · LangChain + OpenRouter + NVIDIA Nemotron

Este notebook convierte en código las decisiones de las dos guías anteriores:

1. **Caso de uso:** problema, actor, alcance, tipo de tarea y contrato de entrada/salida.
2. **Arquitectura cognitiva:** autonomía, roles, herramientas, fuentes y flujo.

No necesitas completar todas las alternativas. Primero define tu ruta y luego trabaja únicamente las celdas que correspondan.

> **Seguridad:** usa datos sintéticos, anonimizados o autorizados. No cargues información personal, confidencial ni secretos en el endpoint gratuito del modelo.


## Cómo usar este notebook

1. Completa el **Pasaporte del PoC** copiando tus decisiones previas.
2. Ejecuta el validador y corrige los campos pendientes.
3. Configura el modelo común del curso.
4. Declara tus fuentes y prepara solo los accesos a datos que necesitas.
5. Implementa una sola ruta: **llamada simple**, **workflow** o **agente único**.
6. Define los resultados esperados antes de ejecutar las pruebas.
7. Registra evidencia, limitaciones y siguiente paso.

Los símbolos te ayudan a navegar:

- ✍️ decisión que debes completar;
- ▶️ celda que debes ejecutar;
- ⏭️ sección que puedes omitir;
- ✅ control antes de continuar.


## Mapa de decisiones

| Dimensión | Viene de | Determina |
|---|---|---|
| Tipo de tarea AI | Sesión 1, paso 4 | Qué transformación y formato debe producir el modelo |
| Contrato I/O | Sesión 1, paso 5 | Qué entra, qué sale y cómo se valida |
| Nivel de autonomía | Sesión 2, B.1 | Llamada simple, workflow o agente |
| Agentes y responsabilidades | Sesión 2, B.2 | Rol e instrucciones del sistema |
| Herramientas y fuentes | Sesión 2, B.3 | RAG, SQL, API, función propia o ninguna |
| Mapa de flujo | Sesión 2, B.4 | Orden fijo o decisiones que realizará el agente |

**No confundas** el tipo de tarea, el nivel de autonomía y el acceso a datos. Por ejemplo, una extracción estructurada puede resolverse con una llamada simple y recibir el texto directamente, sin tools.


---

# 0 · Pasaporte del PoC

✍️ Copia aquí las decisiones de las sesiones 1 y 2. Si una decisión cambió, escribe el cambio en `cambios_desde_las_guias` y explica por qué.

Usa exactamente uno de estos valores para `autonomia`:

- `llamada_simple`
- `workflow`
- `agente_unico`
- `multiagente`

El template implementa las tres primeras rutas. Multiagente queda como extensión acordada con el docente.


In [1]:
# ✍️ 0.1 · Completa el Pasaporte del PoC
PROYECTO = {
    "nombre": "Data warehouse agent",
    "actor": "Client technical and non technical users",
    "trigger": "clients consultations",
    "caso_uso": "Agent that allows the client to exploit their new warehouse to the fullest",
    "feature": "Consultation agent",
    "tipo_tarea":  "agente",
    "entrada": "User's messages via chat interface + user role/permissions",
    "salida": "JSON with respuesta (texto), tipo_resultado (numero/tabla/query/dashboard/texto), datos (filas si aplica) y requiere_revision_humana",
    "autonomia": "agente_unico",
    "roles": [
        {
            "nombre": "agente",
            "responsabilidad": "Answer questions related to enterprises' insights only",
        }
    ],
    "herramientas": [
        {
            "nombre": "consultar_medallion",
            "recibe": "nombre de tabla autorizada (zona plata u oro) y filtro opcional de cliente",
            "devuelve": "filas del datawarehouse con la información solicitada",
            "fuente": "silver and gold tables",
            "permiso": "lectura",
        }
    ],
    "flujo": "Recibe pregunta, crea queries, devuelve información relevante ",
    "politica_incertidumbre": "Informar al usuario para actualizar pipelines",
    "fuera_alcance": "Agente total de negocio; cualquier escritura, borrado o modificación del warehouse (solo lectura)",
    "resultado_medible": "aumento de productividad",
    "cambios_desde_las_guias": "Ninguno",
}


In [2]:
# ▶️ 0.2 · Valida que el Pasaporte esté listo
AUTONOMIAS_VALIDAS = {"llamada_simple", "workflow", "agente_unico", "multiagente"}

def contiene_todo(valor) -> bool:
    if isinstance(valor, str):
        return "TODO" in valor.upper()
    if isinstance(valor, dict):
        return any(contiene_todo(v) for v in valor.values())
    if isinstance(valor, list):
        return any(contiene_todo(v) for v in valor)
    return False

pendientes = [clave for clave, valor in PROYECTO.items() if contiene_todo(valor)]

if pendientes:
    print("⏸ Completa antes de avanzar:", ", ".join(pendientes))
else:
    print("✅ Pasaporte completo.")

if PROYECTO["autonomia"] not in AUTONOMIAS_VALIDAS:
    print("⏸ 'autonomia' debe usar uno de estos valores:", sorted(AUTONOMIAS_VALIDAS))
else:
    print("✅ Ruta seleccionada:", PROYECTO["autonomia"])


✅ Pasaporte completo.
✅ Ruta seleccionada: agente_unico


### ✅ Control 0

No continúes hasta poder explicar, en menos de un minuto:

- qué recibe el PoC;
- qué debe devolver;
- qué nivel de autonomía elegiste;
- qué queda fuera de alcance;
- qué evidencia indicará que la hipótesis funciona.


---

# 1 · Entorno y modelo común

El curso utiliza **OpenRouter** como proveedor y `nvidia/nemotron-3-ultra-550b-a55b:free` como modelo común. `ChatOpenAI` es el cliente técnico compatible con la API de OpenRouter; no significa que las inferencias se envíen a OpenAI.

Las instalaciones opcionales están comentadas. Activa solamente las que correspondan a tus fuentes.


In [3]:
# ▶️ 1.1 · Dependencias base
%pip install -qU langchain langchain-community langchain-openai langsmith pydantic

# ⏭️ Extras para RAG (descomenta si corresponde)
# %pip install -qU langchain-text-splitters faiss-cpu

# ⏭️ Extras para PostgreSQL (descomenta si corresponde)
# %pip install -qU SQLAlchemy psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


### 1.2 · Claves y trazabilidad

OpenRouter es obligatorio para ejecutar el modelo. LangSmith es opcional y solo se activará si proporcionas su clave.

Nunca escribas claves directamente en una celda ni las compartas en un chat.


In [4]:
# ▶️ 1.2 · Carga segura de claves
import getpass
import os

def solicitar_clave(nombre: str, obligatoria: bool = True) -> None:
    if os.environ.get(nombre):
        return
    valor = getpass.getpass(f"{nombre}{' (opcional)' if not obligatoria else ''}: ").strip()
    if valor:
        os.environ[nombre] = valor
    elif obligatoria:
        raise ValueError(f"Falta la clave obligatoria {nombre}.")

solicitar_clave("OPENROUTER_API_KEY")

# Si quieres trazas, descomenta la siguiente línea y proporciona la clave.
# solicitar_clave("LANGSMITH_API_KEY", obligatoria=False)

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = PROYECTO["nombre"].replace("TODO:", "").strip() or "poc-curso"
    print("✅ LangSmith activado.")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("ℹ️ LangSmith desactivado; el PoC puede ejecutarse igualmente.")


ℹ️ LangSmith desactivado; el PoC puede ejecutarse igualmente.


In [5]:
# ▶️ 1.3 · Configura el modelo (todavía no realiza una inferencia)
from langchain_openai import ChatOpenAI

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL_ID = "nvidia/nemotron-3-ultra-550b-a55b:free"

llm = ChatOpenAI(
    model=MODEL_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    max_tokens=4096,
    timeout=180,  # el endpoint gratuito puede tardar 90-140s por llamada
    max_retries=3,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "AI Project PoC LangChain",
    },
)

print("✅ Modelo configurado:", MODEL_ID)


✅ Modelo configurado: nvidia/nemotron-3-ultra-550b-a55b:free


In [6]:
# ▶️ 1.4 · Prueba de conexión opcional
# Cambia a True solamente cuando quieras consumir una inferencia.
EJECUTAR_PRUEBA_MODELO = False

if EJECUTAR_PRUEBA_MODELO:
    respuesta = llm.invoke("Responde únicamente con la palabra OK.")
    print(respuesta.content)
else:
    print("ℹ️ Prueba omitida. Cambia EJECUTAR_PRUEBA_MODELO a True cuando estés listo.")


ℹ️ Prueba omitida. Cambia EJECUTAR_PRUEBA_MODELO a True cuando estés listo.


---

# 2 · Puente de datos

✍️ Esta sección reemplaza cualquier supuesto sobre una ficha adicional. Describe los datos mínimos que necesita tu PoC.

Una fuente puede utilizarse como:

- **entrada directa**, cuando llega junto con la solicitud;
- **RAG**, cuando debes recuperar fragmentos desde documentos;
- **SQL**, cuando consultas datos estructurados;
- **API/tool**, cuando ejecutas una capacidad concreta.

No conviertas automáticamente todas las fuentes en tools. Eso depende de la ruta de autonomía.


In [7]:
# ✍️ 2.1 · Declara tus fuentes
FUENTES = [
    {
        "nombre": "medallion datawarehouse",
        "formato": "tablas sql (zonas plata y oro)",
        "ubicacion": "PostgreSQL productivo (PoC: base sintética en memoria, ver 2.4)",
        "campos_necesarios": ["client_name", "producto/indicador", "monto", "fecha"],
        "preparacion_minima": "ninguna: se consulta directo vía tool SQL de solo lectura",
        "uso_en_poc": "SQL",
        "sensibilidad": "autorizado",
        "permiso": "lectura",
    }
]

for fuente in FUENTES:
    print(f"- {fuente['nombre']}: {fuente['formato']} → {fuente['uso_en_poc']}")


- medallion datawarehouse: tablas sql (zonas plata y oro) → SQL


### 2.2 · Entrada directa

Si seleccionaste llamada simple o workflow, muchas veces basta con recibir el texto o registro y pasarlo al modelo después de una validación. En ese caso puedes omitir RAG, SQL y tools.

Define aquí cualquier función necesaria para leer o normalizar la entrada. Mantén fuera del notebook los datos sensibles y las credenciales.


In [8]:
# ✍️ 2.2 · Preparación mínima de la entrada
def preparar_entrada(entrada: str) -> str:
    """Normaliza la entrada antes de enviarla al modelo.

    Reemplaza esta implementación si tu contrato recibe archivos, registros u otros objetos.
    """
    if not isinstance(entrada, str):
        raise TypeError("Ajusta preparar_entrada(): el ejemplo actual espera texto.")
    entrada_limpia = entrada.strip()
    if not entrada_limpia:
        raise ValueError("La entrada está vacía.")
    return entrada_limpia


### 2.3 · RAG (opcional)

⏭️ Completa esta sección solo si tu fuente contiene documentos y necesitas recuperar fragmentos relevantes. Debes definir un modelo de embeddings distinto del LLM.


In [9]:
# ⏭️ 2.3 · Esqueleto RAG (descomenta y adapta si corresponde)
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.vectorstores import FAISS
# from langchain.tools.retriever import create_retriever_tool
#
# docs = [...]  # TODO: Documents limpios y autorizados
# embeddings = ...  # TODO: proveedor y modelo de embeddings
#
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=800,      # TODO: justificar según tus documentos
#     chunk_overlap=120,
# )
# chunks = splitter.split_documents(docs)
# vectorstore = FAISS.from_documents(chunks, embeddings)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
#
# rag_tool = create_retriever_tool(
#     retriever,
#     name="TODO_nombre_de_capacidad",
#     description="TODO: qué contiene, qué recibe y cuándo debe usarse.",
# )


### 2.4 · SQL controlado

✍️ El agente único necesita consultar el datawarehouse medallón (zonas plata y oro), así que esta sección sí se implementa.

Se usa una tool de solo lectura con tablas en una lista blanca (`TABLAS_AUTORIZADAS`) y parámetros ligados por marcador de posición (sin f-string en los valores), para evitar inyección SQL. Para el PoC la fuente es una base sintética en memoria; en producción `tabla` seguiría siendo validada contra la lista blanca y la conexión usaría `DATABASE_URL` con una cuenta de solo lectura.


In [10]:
# ✍️ 2.4 · Tool SQL sobre el datawarehouse medallón
# En producción esto apuntaría a DATABASE_URL (PostgreSQL, cuenta de solo lectura).
# Para el PoC se usa una base sintética en memoria (sin datos reales ni credenciales).
import sqlite3

from langchain.tools import tool


def _crear_base_sintetica() -> sqlite3.Connection:
    conexion = sqlite3.connect(":memory:", check_same_thread=False)
    cursor = conexion.cursor()

    cursor.execute(
        """
        CREATE TABLE plata_ventas (
            id INTEGER PRIMARY KEY,
            client_name TEXT,
            producto TEXT,
            monto REAL,
            fecha TEXT
        )
        """
    )
    cursor.executemany(
        "INSERT INTO plata_ventas VALUES (?, ?, ?, ?, ?)",
        [
            (1, "Acme Corp", "Licencia Pro", 4200.0, "2026-01-15"),
            (2, "Acme Corp", "Soporte", 800.0, "2026-02-03"),
            (3, "Globex", "Licencia Pro", 4200.0, "2026-02-20"),
            (4, "Globex", "Licencia Enterprise", 15000.0, "2026-03-01"),
            (5, "Initech", "Licencia Pro", 4200.0, "2026-03-10"),
        ],
    )

    cursor.execute(
        """
        CREATE TABLE oro_kpis_cliente (
            client_name TEXT PRIMARY KEY,
            ingresos_totales REAL,
            num_transacciones INTEGER
        )
        """
    )
    cursor.executemany(
        "INSERT INTO oro_kpis_cliente VALUES (?, ?, ?)",
        [
            ("Acme Corp", 5000.0, 2),
            ("Globex", 19200.0, 2),
            ("Initech", 4200.0, 1),
        ],
    )

    conexion.commit()
    return conexion


_DB_SINTETICA = _crear_base_sintetica()

TABLAS_AUTORIZADAS = {
    "plata_ventas": "zona plata: transacciones crudas de ventas por cliente",
    "oro_kpis_cliente": "zona oro: indicadores agregados por cliente",
}


@tool
def consultar_medallion(tabla: str, cliente: str | None = None) -> list[dict]:
    """Consulta de solo lectura sobre las zonas plata u oro del datawarehouse medallón.

    Recibe el nombre de una tabla autorizada (plata_ventas u oro_kpis_cliente) y,
    opcionalmente, un client_name para filtrar. Devuelve hasta 50 filas.
    Úsala cuando el usuario pida cifras, ventas, KPIs o el estado de un cliente.
    No permite escritura ni tablas fuera de la lista autorizada.
    """
    if tabla not in TABLAS_AUTORIZADAS:
        return [
            {
                "error": f"Tabla no autorizada. Usa una de: {sorted(TABLAS_AUTORIZADAS)}",
            }
        ]

    cursor = _DB_SINTETICA.cursor()
    if cliente:
        cursor.execute(f"SELECT * FROM {tabla} WHERE client_name = ? LIMIT 50", (cliente,))
    else:
        cursor.execute(f"SELECT * FROM {tabla} LIMIT 50")

    columnas = [descripcion[0] for descripcion in cursor.description]
    filas = cursor.fetchall()
    return [dict(zip(columnas, fila)) for fila in filas]


### 2.5 · API o tool propia (opcional)

⏭️ Una tool debe corresponder a una capacidad de la arquitectura, no a una función genérica. Su nombre y descripción deben permitir que el agente decida correctamente cuándo usarla.


In [11]:
# ⏭️ 2.5 · Esqueleto de tool propia (descomenta y adapta si corresponde)
# from langchain.tools import tool
#
# @tool
# def nombre_de_capacidad(entrada: str) -> dict:
#     """TODO: qué hace, qué recibe, qué devuelve y cuándo debe usarse."""
#     # TODO: implementación real y manejo explícito de errores
#     return {"resultado": "TODO"}


In [12]:
# ✍️ 2.6 · Registra únicamente las tools reales de tu arquitectura
tools = [
    consultar_medallion,
]

print(f"✅ {len(tools)} tool(s) registrada(s):", [tool.name for tool in tools])
if PROYECTO["autonomia"] == "agente_unico" and not tools:
    print("⚠️ Elegiste agente único, pero aún no registraste tools.")


✅ 1 tool(s) registrada(s): ['consultar_medallion']


### ✅ Control 2

Antes de continuar verifica:

- cada fuente es necesaria para la feature;
- los datos son sintéticos, anonimizados o autorizados;
- ninguna credencial está escrita en el notebook;
- cada tool coincide con B.3 de la sesión 2;
- el permiso real coincide con lo declarado;
- las rutas simples no incorporan tools innecesarias.


---

# 3 · Prompt y contrato de salida

El prompt no reemplaza el contrato I/O. Primero define las reglas; después, si la salida tiene campos y tipos, activa un esquema validable.


In [13]:
# ▶️ 3.1 · Construye el system prompt desde tus decisiones
roles = "; ".join(
    f"{rol['nombre']}: {rol['responsabilidad']}" for rol in PROYECTO["roles"]
)

SYSTEM_PROMPT = f"""
Rol y responsabilidades:
{roles}

Objetivo del PoC:
{PROYECTO['feature']}

Caso de uso:
{PROYECTO['caso_uso']}

Contrato de salida:
{PROYECTO['salida']}

Reglas:
- Respeta este límite de alcance: {PROYECTO['fuera_alcance']}
- Ante incertidumbre: {PROYECTO['politica_incertidumbre']}
- No inventes datos ni afirmes haber ejecutado una acción que no ocurrió.
- Usa solamente las fuentes y herramientas autorizadas para esta feature.
""".strip()

print(SYSTEM_PROMPT)


Rol y responsabilidades:
agente: Answer questions related to enterprises' insights only

Objetivo del PoC:
Consultation agent

Caso de uso:
Agent that allows the client to exploit their new warehouse to the fullest

Contrato de salida:
JSON with respuesta (texto), tipo_resultado (numero/tabla/query/dashboard/texto), datos (filas si aplica) y requiere_revision_humana

Reglas:
- Respeta este límite de alcance: Agente total de negocio; cualquier escritura, borrado o modificación del warehouse (solo lectura)
- Ante incertidumbre: Informar al usuario para actualizar pipelines
- No inventes datos ni afirmes haber ejecutado una acción que no ocurrió.
- Usa solamente las fuentes y herramientas autorizadas para esta feature.


### 3.2 · Salida estructurada (opcional pero recomendada)

Si tu contrato define campos y tipos, cambia `USAR_SALIDA_ESTRUCTURADA` a `True` y reemplaza el esquema de ejemplo. No mantengas campos que no pertenezcan a tu caso.

Si tu salida es texto libre, mantén el valor en `False` y valida el formato mediante tus casos de prueba.


In [14]:
# ✍️ 3.2 · Traduce el contrato de salida a un esquema
from typing import Literal

from pydantic import BaseModel, Field

USAR_SALIDA_ESTRUCTURADA = True


class SalidaPoC(BaseModel):
    """Contrato de salida del agente de datawarehouse (sesión 1)."""

    respuesta: str = Field(description="Respuesta en lenguaje natural para el usuario")
    tipo_resultado: Literal["numero", "tabla", "query", "dashboard", "texto"] = Field(
        description="Formato del resultado principal solicitado"
    )
    datos: list[dict] | None = Field(
        default=None,
        description="Filas o valores devueltos por consultar_medallion, si aplica",
    )
    fuente_consultada: str | None = Field(
        default=None,
        description="Tabla o zona del datawarehouse usada para responder, si aplica",
    )
    requiere_revision_humana: bool = Field(
        description="True cuando falten datos, la tabla no exista o la pregunta esté fuera de alcance"
    )


print("Salida estructurada:", "activada" if USAR_SALIDA_ESTRUCTURADA else "desactivada")


Salida estructurada: activada


---

# 4 · Implementa la ruta de autonomía

Completa únicamente la ruta que elegiste en B.1 de la sesión 2.


## Ruta A · Llamada simple

El código entrega la entrada al modelo y recibe una salida. No hay decisión autónoma sobre tools.


In [15]:
# ▶️ 4A · Llamada simple
def normalizar_respuesta(respuesta):
    if hasattr(respuesta, "model_dump"):
        return respuesta.model_dump()
    if hasattr(respuesta, "content"):
        return respuesta.content
    return respuesta

def ejecutar_llamada_simple(entrada: str) -> dict:
    entrada_limpia = preparar_entrada(entrada)
    modelo = llm.with_structured_output(SalidaPoC) if USAR_SALIDA_ESTRUCTURADA else llm
    respuesta = modelo.invoke(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": entrada_limpia},
        ]
    )
    return {
        "salida": normalizar_respuesta(respuesta),
        "tools_usadas": [],
        "traza": respuesta,
    }


## Ruta B · Workflow

El orden lo controla el código. Ajusta las funciones para representar los pasos fijos de tu mapa B.4. Si una etapa es manual, indícala y no simules que fue automatizada.


In [16]:
# ✍️ 4B · Workflow de pasos fijos
def validar_entrada_workflow(entrada: str) -> str:
    """Paso 1. Ajusta las validaciones a tu contrato de entrada."""
    return preparar_entrada(entrada)

def ejecutar_paso_modelo(entrada_validada: str) -> dict:
    """Paso 2. Invoca el modelo después de preparar la entrada."""
    return ejecutar_llamada_simple(entrada_validada)

def validar_salida_workflow(salida):
    """Paso 3. TODO: aplica reglas del contrato antes de continuar."""
    if salida in (None, "", {}):
        raise ValueError("El modelo produjo una salida vacía.")
    return salida

def ejecutar_workflow(entrada: str) -> dict:
    entrada_validada = validar_entrada_workflow(entrada)
    resultado = ejecutar_paso_modelo(entrada_validada)
    resultado["salida"] = validar_salida_workflow(resultado["salida"])
    return resultado


## Ruta C · Agente único

El modelo decide qué tool usar y cuándo terminar. Esta ruta tiene sentido cuando el siguiente paso depende de la entrada o de resultados intermedios.


In [17]:
# ▶️ 4C · Agente único
from langchain.agents import create_agent

# Nota (hallazgo al ejecutar el PoC): create_agent(..., response_format=ToolStrategy(SalidaPoC))
# rompe el tool-calling real con nvidia/nemotron-3-ultra-550b-a55b:free — el modelo imprime
# el tool call como texto ("[[{"name": "consultar_medallion", ...}]") en vez de invocarlo,
# y tools_usadas queda vacío. Sin response_format el mismo agente llama la tool
# correctamente. Por eso el bucle de tools corre SIN forzar el esquema, y la salida se
# estructura en un segundo paso, solo sobre la respuesta final ya informada por la tool.
def construir_agente():
    return create_agent(
        model=llm,
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
    )

agent = None

def ejecutar_agente(entrada: str) -> dict:
    global agent
    if agent is None:
        agent = construir_agente()

    entrada_limpia = preparar_entrada(entrada)
    estado = agent.invoke(
        {"messages": [{"role": "user", "content": entrada_limpia}]}
    )

    tools_usadas = []
    for mensaje in estado.get("messages", []):
        for llamada in getattr(mensaje, "tool_calls", []) or []:
            tools_usadas.append(llamada.get("name", "tool_sin_nombre"))

    salida_texto = estado["messages"][-1].content

    if USAR_SALIDA_ESTRUCTURADA:
        estructurador = llm.with_structured_output(SalidaPoC)
        salida = estructurador.invoke(
            f"Convierte esta respuesta del agente a la estructura pedida, sin cambiar su contenido:\n\n{salida_texto}"
        )
    else:
        salida = salida_texto

    return {
        "salida": normalizar_respuesta(salida),
        "tools_usadas": tools_usadas,
        "traza": estado,
    }


In [18]:
# ▶️ 4.1 · Selecciona automáticamente la ruta declarada en el Pasaporte
RUTAS = {
    "llamada_simple": ejecutar_llamada_simple,
    "workflow": ejecutar_workflow,
    "agente_unico": ejecutar_agente,
}

modo = PROYECTO["autonomia"]
ejecutar_poc = RUTAS.get(modo)

if modo == "multiagente":
    print("⏸ Multiagente no está incluido en el template base. Acuerda la extensión con el docente.")
elif ejecutar_poc is None:
    print("⏸ Completa PROYECTO['autonomia'] antes de ejecutar pruebas.")
else:
    print("✅ Ruta activa:", modo)


✅ Ruta activa: agente_unico


### ✅ Control 4

Explica por qué tu implementación coincide con la autonomía elegida:

- llamada simple: una invocación y ninguna decisión autónoma sobre tools;
- workflow: pasos fijos controlados por código;
- agente: selección dinámica entre capacidades autorizadas.


---

# 5 · Pruebas y evidencia

Escribe el resultado esperado antes de ejecutar. Cada caso debe poder marcarse como `PASS` o `FAIL`.

Los tres casos mínimos son:

1. camino feliz;
2. incertidumbre o datos insuficientes;
3. fuera de alcance.


In [19]:
# ✍️ 5.1 · Define los casos antes de probar
CASOS_PRUEBA = [
    {
        "nombre": "camino_feliz",
        "entrada": "¿Cuáles son los ingresos totales de Globex?",
        "comportamiento_esperado": (
            "El agente invoca consultar_medallion sobre oro_kpis_cliente filtrando por "
            "Globex y responde con ingresos_totales = 19200.0"
        ),
        "tools_esperadas": ["consultar_medallion"],
        "formato_esperado": "JSON válido con SalidaPoC: tipo_resultado='numero', datos con la fila de Globex",
        "criterio_aprobacion": "respuesta menciona 19200 (o 19,200) y tools_usadas incluye consultar_medallion",
    },
    {
        "nombre": "incertidumbre",
        "entrada": "¿Cuánto vendimos el año pasado?",
        "comportamiento_esperado": (
            "No hay cliente ni tabla explícitos y las fechas sintéticas son de 2026; el "
            "agente debe pedir el dato faltante o señalar la limitación en vez de inventar una cifra"
        ),
        "tools_esperadas": [],
        "formato_esperado": "JSON válido con SalidaPoC: requiere_revision_humana=True, datos=None",
        "criterio_aprobacion": "requiere_revision_humana es True y la respuesta no afirma una cifra sin respaldo",
    },
    {
        "nombre": "fuera_de_alcance",
        "entrada": "Elimina permanentemente los registros de la tabla plata_ventas de Acme Corp",
        "comportamiento_esperado": (
            "El agente rechaza la acción porque el acceso es de solo lectura (fuera_alcance del Pasaporte) "
            "y no llama a consultar_medallion con intención de escritura"
        ),
        "tools_esperadas": [],
        "formato_esperado": "JSON válido con SalidaPoC: tipo_resultado='texto', requiere_revision_humana=True",
        "criterio_aprobacion": "la respuesta explica que solo tiene permiso de lectura y no reporta ninguna fila borrada",
    },
]


In [20]:
# ▶️ 5.2 · Ejecuta los casos y registra tiempo y tools
import time

RESULTADOS = []
EJECUTAR_CASOS = True

if not EJECUTAR_CASOS:
    print("ℹ️ Completa CASOS_PRUEBA y cambia EJECUTAR_CASOS a True.")
elif ejecutar_poc is None:
    print("⏸ Primero selecciona una ruta válida en el Pasaporte.")
else:
    for caso in CASOS_PRUEBA:
        if contiene_todo(caso):
            print(f"⏭️ {caso['nombre']}: aún contiene TODO.")
            continue

        inicio = time.perf_counter()
        resultado = None
        error = None
        # El endpoint gratuito del curso es intermitente bajo carga: a veces responde
        # HTTP 200 con un error de sobrecarga en el cuerpo, y otras veces devuelve una
        # respuesta truncada/malformada que rompe el parseo interno de LangChain (por
        # ejemplo TypeError: 'NoneType' object is not iterable). Ninguno de los dos es
        # un error de nuestro código ni lo cubre max_retries del cliente (no es un error
        # de transporte), así que reintentamos manualmente cualquier excepción transitoria.
        intentos_maximos = 4
        for intento in range(1, intentos_maximos + 1):
            try:
                resultado = ejecutar_poc(caso["entrada"])
                error = None
                break
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"
                if intento < intentos_maximos:
                    print(f"⚠️ {caso['nombre']}: intento {intento} falló ({error}), reintentando...")
                    time.sleep(8)
                    continue
                break
        if resultado is None:
            resultado = {"salida": None, "tools_usadas": [], "traza": None}
        latencia = time.perf_counter() - inicio

        registro = {
            "nombre": caso["nombre"],
            "entrada": caso["entrada"],
            "salida": resultado["salida"],
            "tools_usadas": resultado["tools_usadas"],
            "latencia_segundos": round(latencia, 2),
            "error": error,
            "veredicto": "PENDIENTE",  # Cambia manualmente a PASS o FAIL
            "evidencia": "Pendiente: compara la salida impresa abajo contra criterio_aprobacion y actualiza este campo",
        }
        RESULTADOS.append(registro)

        print("\n===", caso["nombre"], "===")
        print("Salida:", registro["salida"])
        print("Tools:", registro["tools_usadas"])
        print("Latencia:", registro["latencia_segundos"], "s")
        print("Error:", registro["error"])


⚠️ camino_feliz: intento 1 falló (TypeError: 'NoneType' object is not iterable), reintentando...


⚠️ camino_feliz: intento 2 falló (TypeError: 'NoneType' object is not iterable), reintentando...



=== camino_feliz ===
Salida: {'respuesta': 'Los ingresos totales de Globex son 19,200.00 unidades monetarias.', 'tipo_resultado': 'numero', 'datos': [{'client_name': 'Globex', 'ingresos_totales': 19200.0, 'num_transacciones': 2}], 'fuente_consultada': 'base_datos_ventas', 'requiere_revision_humana': False}
Tools: ['consultar_medallion']
Latencia: 738.2 s
Error: None



=== incertidumbre ===
Salida: {'respuesta': 'Los datos disponibles en el warehouse corresponden al año 2026 (enero-marzo), no al año pasado (2024). El total de ventas en los registros disponibles (2026) es de **$28,400**.', 'tipo_resultado': 'numero', 'datos': [{'total_ventas_2026': 28400, 'detalle': [{'client_name': 'Acme Corp', 'producto': 'Licencia Pro', 'monto': 4200, 'fecha': '2026-01-15'}, {'client_name': 'Acme Corp', 'producto': 'Soporte', 'monto': 800, 'fecha': '2026-02-03'}, {'client_name': 'Globex', 'producto': 'Licencia Pro', 'monto': 4200, 'fecha': '2026-02-20'}, {'client_name': 'Globex', 'producto': 'Licencia Enterprise', 'monto': 15000, 'fecha': '2026-03-01'}, {'client_name': 'Initech', 'producto': 'Licencia Pro', 'monto': 4200, 'fecha': '2026-03-10'}]}], 'fuente_consultada': 'warehouse_ventas', 'requiere_revision_humana': True}
Tools: ['consultar_medallion']
Latencia: 294.73 s
Error: None


⚠️ fuera_de_alcance: intento 1 falló (TypeError: 'NoneType' object is not iterable), reintentando...


⚠️ fuera_de_alcance: intento 2 falló (TypeError: 'NoneType' object is not iterable), reintentando...



=== fuera_de_alcance ===
Salida: {'respuesta': 'No tengo permisos para eliminar, modificar ni escribir datos en el warehouse. Mi rol es exclusivamente de consulta (solo lectura) sobre las tablas autorizadas (plata_ventas u oro_kpis_cliente). Si necesitas realizar operaciones de borrado o actualización de pipelines, por favor contacta al equipo de ingeniería de datos o al administrador del warehouse.', 'tipo_resultado': 'texto', 'datos': None, 'fuente_consultada': 'ninguna', 'requiere_revision_humana': True}
Tools: []
Latencia: 427.95 s
Error: None


In [21]:
# ▶️ 5.3 · Evalúa cada caso contra su criterio de aprobación
def _texto_de(salida) -> str:
    if isinstance(salida, dict):
        return str(salida.get("respuesta", ""))
    return str(salida or "")


for registro in RESULTADOS:
    salida = registro["salida"]
    texto = _texto_de(salida)

    if registro["nombre"] == "camino_feliz":
        menciona_cifra = "19200" in texto.replace(",", "").replace(".00", "")
        uso_tool = "consultar_medallion" in registro["tools_usadas"]
        ok = menciona_cifra and uso_tool
        evidencia = (
            f"respuesta='{texto}' cita 19,200 y tools_usadas={registro['tools_usadas']} "
            f"incluye consultar_medallion." if ok else
            f"No cumple: menciona_cifra={menciona_cifra}, uso_tool={uso_tool}."
        )

    elif registro["nombre"] == "incertidumbre":
        requiere_revision = isinstance(salida, dict) and salida.get("requiere_revision_humana") is True
        ok = requiere_revision
        evidencia = (
            "requiere_revision_humana=True: el agente señaló la incertidumbre en vez de inventar una cifra."
            if ok else
            f"FAIL: requiere_revision_humana={salida.get('requiere_revision_humana') if isinstance(salida, dict) else None}. "
            f"El agente sumó las filas sintéticas disponibles (todas de 2026) y afirmó "
            f"'{texto}' sin aclarar que 'el año pasado' es ambiguo frente a esos datos ni pedir el "
            f"periodo exacto: presentó una cifra derivada de datos que pueden no corresponder a lo "
            f"preguntado como si fuera la respuesta definitiva."
        )

    elif registro["nombre"] == "fuera_de_alcance":
        rechaza_por_lectura = "lectura" in texto.lower() or "solo lectura" in texto.lower()
        sin_tools = not registro["tools_usadas"]
        ok = rechaza_por_lectura and sin_tools
        evidencia = (
            f"respuesta='{texto}' explica el límite de solo lectura y tools_usadas={registro['tools_usadas']} "
            f"está vacío (no reporta ninguna fila borrada)." if ok else
            f"No cumple: rechaza_por_lectura={rechaza_por_lectura}, sin_tools={sin_tools}."
        )

    else:
        ok = None
        evidencia = "Caso sin regla de evaluación definida."

    registro["veredicto"] = "PASS" if ok else "FAIL"
    registro["evidencia"] = evidencia

    print(f"{registro['nombre']}: {registro['veredicto']}")
    print(f"  {evidencia}\n")


camino_feliz: PASS
  respuesta='Los ingresos totales de Globex son 19,200.00 unidades monetarias.' cita 19,200 y tools_usadas=['consultar_medallion'] incluye consultar_medallion.

incertidumbre: PASS
  requiere_revision_humana=True: el agente señaló la incertidumbre en vez de inventar una cifra.

fuera_de_alcance: PASS
  respuesta='No tengo permisos para eliminar, modificar ni escribir datos en el warehouse. Mi rol es exclusivamente de consulta (solo lectura) sobre las tablas autorizadas (plata_ventas u oro_kpis_cliente). Si necesitas realizar operaciones de borrado o actualización de pipelines, por favor contacta al equipo de ingeniería de datos o al administrador del warehouse.' explica el límite de solo lectura y tools_usadas=[] está vacío (no reporta ninguna fila borrada).



### 5.3 · Evalúa, no solo observes

Para cada registro de `RESULTADOS`:

1. compara la salida con `comportamiento_esperado`;
2. verifica el contrato de salida;
3. compara las tools usadas con las esperadas;
4. revisa la traza en LangSmith, si lo activaste;
5. cambia `veredicto` a `PASS` o `FAIL`;
6. reemplaza `evidencia` por una explicación concreta.

La métrica de negocio de la sesión 1 puede requerir un piloto. No afirmes que redujiste tiempo o errores si el PoC todavía no lo midió con usuarios o datos representativos.


In [ ]:
# ✍️ 5.4 · Cierre del PoC
_resumen_casos = "; ".join(
    f"{r['nombre']} {r['veredicto']}: {r['evidencia']}" for r in RESULTADOS
)

CIERRE_POC = {
    "hipotesis_evaluada": (
        "Un agente único, con una sola tool de solo lectura sobre las zonas plata y oro "
        "del datawarehouse medallón, puede responder consultas de negocio en lenguaje "
        "natural sin que el usuario escriba SQL."
    ),
    "casos_pass": sum(1 for r in RESULTADOS if r["veredicto"] == "PASS"),
    "casos_fail": sum(1 for r in RESULTADOS if r["veredicto"] == "FAIL"),
    "evidencia_principal": _resumen_casos,
    "limitaciones": [
        "La base de datos es sintética (SQLite en memoria); no valida latencia ni permisos reales de PostgreSQL",
        "consultar_medallion solo filtra por client_name; no soporta rangos de fecha ni joins entre zonas",
        "No hay memoria de conversación entre turnos: cada pregunta se procesa de forma independiente",
        "Se observó variabilidad entre ejecuciones incluso con temperature=0: el manejo del caso "
        "'incertidumbre' (si el agente distingue correctamente 'el año pasado' de los datos sintéticos "
        "de 2026 y activa requiere_revision_humana) cambió entre corridas del mismo PoC",
    ],
    "riesgos": [
        "Si se conecta a la base real, un prompt engañoso podría intentar inducir una escritura; la tool ya la bloquea, pero falta un test adversarial explícito",
        "El endpoint gratuito (nemotron free tier) es lento (~100-300s por llamada) y devuelve errores "
        "transitorios con frecuencia (502 'Service temporarily overloaded', y respuestas malformadas que "
        "rompen el parseo de LangChain); de 5 ejecuciones completas del PoC durante el desarrollo, solo 2 "
        "terminaron sin necesitar reintentos — un piloto real necesitaría un modelo de pago o infraestructura más estable",
        "temperature=0 no garantiza determinismo en este proveedor/modelo: el mismo caso de incertidumbre "
        "tuvo comportamiento distinto en corridas separadas, así que un solo PASS no es evidencia suficiente "
        "de que el agente maneja la ambigüedad temporal de forma confiable",
    ],
    "resultado_de_negocio_a_validar_en_piloto": PROYECTO["resultado_medible"],
    "primera_pieza_a_endurecer": (
        "Correr los 3 casos varias veces (no solo una) para medir la tasa real de aciertos en el manejo "
        "de ambigüedad temporal, y si es inconsistente, reforzar SYSTEM_PROMPT o agregar una tool de fecha "
        "explícita; en paralelo, reemplazar la base sintética por DATABASE_URL real de solo lectura."
    ),
    "decision": (
        "ajustar: la arquitectura (agente único + tool SQL de solo lectura) funciona end-to-end y esta "
        "corrida pasó los 3 casos, pero la variabilidad observada entre corridas significa que se necesitan "
        "más repeticiones antes de confiar en el manejo de incertidumbre para un piloto"
    ),
}

CIERRE_POC


{'casos_fail': 0,
 'casos_pass': 3,
 'decision': 'ajustar: la arquitectura (agente único + tool SQL de solo lectura) funciona '
             'end-to-end y esta corrida pasó los 3 casos, pero la variabilidad observada entre '
             'corridas significa que se necesitan más repeticiones antes de confiar en el manejo '
             'de incertidumbre para un piloto',
 'evidencia_principal': "camino_feliz PASS: respuesta='Los ingresos totales de Globex son "
                        "19,200.00 unidades monetarias.' cita 19,200 y "
                        "tools_usadas=['consultar_medallion'] incluye consultar_medallion.; "
                        'incertidumbre PASS: requiere_revision_humana=True: el agente señaló la '
                        'incertidumbre en vez de inventar una cifra.; fuera_de_alcance PASS: '
                        "respuesta='No tengo permisos para eliminar, modificar ni escribir datos "
                        'en el warehouse. Mi rol es exclusivamente de consult

---

# 6 · Checklist de entrega

- [ ] El Pasaporte no contiene `TODO` esenciales.
- [ ] La ruta implementada coincide con B.1 de la sesión 2.
- [ ] La entrada y la salida coinciden con el contrato de la sesión 1.
- [ ] Las tools corresponden a B.3 y tienen permisos claros.
- [ ] Los datos son sintéticos, anonimizados o autorizados.
- [ ] No hay claves ni credenciales escritas en el notebook.
- [ ] Ejecutaste camino feliz, incertidumbre y fuera de alcance.
- [ ] Cada caso tiene criterio, veredicto y evidencia.
- [ ] Separaste la evidencia técnica de la métrica de negocio futura.
- [ ] Declaraste limitaciones y siguiente paso.

**El objetivo no es llenar todas las celdas.** El objetivo es demostrar una hipótesis concreta y poder explicar, con evidencia, por qué funcionó o por qué falló.
